In [1]:
import sys
import time
import math

# Fisrt, we add the location of the library to test to the PYTHON path
from liblibra_core import *
from libra_py import units as units
from libra_py import influence_spectrum as infsp
from libra_py import data_visualize
from libra_py import data_conv
from libra_py import data_stat
import libra_py.workflows.nbra.step4 as step4
import libra_py.workflows.nbra.decoherence_times as decoherence_times
import numpy as np
import multiprocessing as mp
import matplotlib.pyplot as plt
import os


<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

In [ ]:
####################
# 1. Read / Get the Hvib files from step3/res_mb_sp
# For the CI (MB) case 
print ("\nGathering data from MD ")
#absolute_path = os.getcwd()
#params = {}
#params["data_set_paths"] = []
#params["data_set_paths"].append(absolute_path+"/3_step3/ci_zn-cspbi3/")
#params["Hvib_re_prefix"] = "hvib_adi_"; params["Hvib_re_suffix"] = ".txt"
#params["Hvib_im_prefix"] = ""; params["Hvib_im_suffix"] = ""
#params["nfiles"]         = 999
#params["nstates"]        = 11 # total number of electronic states
#params["init_times"]     = [0]
#params["active_space"]   = list(range(params["nstates"])) # indexing is from 0!
# Include HOMO and up to the last electronic state
#hvib_mb = step4.get_Hvib2(params)
#hvib_mb[0][-1].show_matrix()
#print ("Length of hvib_mb is: ", len(hvib_mb[0]))
#sys.exit(0)



In [5]:
# For the SD (SP) case
import os
import re
from liblibra_core import CMATRIX

print("\nGathering data from MD")

absolute_path = os.getcwd()
data_path = os.path.join(absolute_path, "3_step3", "ci_zn-cspbi3")

nfiles = 999
nstates = 11

# Regular expression for (real,imag)
pattern = re.compile(r"\(([-+0-9Ee\.]+),([-+0-9Ee\.]+)\)")

hvib_mixed_sd = []

for step in range(nfiles):

    filename = os.path.join(data_path, f"hvib_adi_{step}.txt")

    H = CMATRIX(nstates, nstates)

    with open(filename, "r") as f:
        for i, line in enumerate(f):
            vals = pattern.findall(line)

            for j, (re_part, im_part) in enumerate(vals):
                H.set(i, j, float(re_part), float(im_part))

    hvib_mixed_sd.append(H)

print("Number of Hvib matrices =", len(hvib_mixed_sd))

hvib_mixed_sd[-1].show_matrix()


Gathering data from MD
Number of Hvib matrices = 999
(0.0000000,0.0000000)  (0.0000000,-0.00044880739)  (0.0000000,-0.00069705378)  (0.0000000,-0.0010796118)  (0.0000000,-0.00030796219)  (0.0000000,0.00039085843)  (0.0000000,0.0010199902)  (0.0000000,-0.00056891974)  (0.0000000,0.00062270449)  (0.0000000,0.0012393884)  (0.0000000,-0.00011659686)  
(0.0000000,0.00044880739)  (0.061791739,0.0000000)  (0.0000000,-0.00011129230)  (0.0000000,-0.00046618636)  (0.0000000,-0.0010987561)  (0.0000000,-0.00043595946)  (0.0000000,0.00055801717)  (0.0000000,0.00023998565)  (0.0000000,0.00055154789)  (0.0000000,-2.1131279e-05)  (0.0000000,-9.6273296e-05)  
(0.0000000,0.00069705378)  (0.0000000,0.00011129230)  (0.066519606,0.0000000)  (0.0000000,2.7447950e-05)  (0.0000000,-0.0014005216)  (0.0000000,-0.00024546027)  (0.0000000,0.00027840949)  (0.0000000,0.00057173164)  (0.0000000,0.00027539388)  (0.0000000,-0.00099858521)  (0.0000000,0.00045079987)  
(0.0000000,0.0010796118)  (0.0000000,0.00046618636

In [6]:
####################
# 2. Define some helper functions. Read each function for more detail

import numpy as np
from libra_py import data_stat
from libra_py import units


def compute_state_energies_vs_time(hvib):
    """
    Compute adiabatic state energies relative to state 0.

    Parameters
    ----------
    hvib : list of CMATRIX
        Vibronic Hamiltonians for all MD time steps.

    Returns
    -------
    energies : ndarray
        Shape = (nstates, nsteps)
        Energies in Hartree relative to state 0.
    """

    nsteps = len(hvib)
    nstates = hvib[0].num_of_rows

    energies = np.zeros((nstates, nsteps))

    for step in range(nsteps):

        E0 = hvib[step].get(0, 0).real

        for state in range(nstates):
            energies[state, step] = (
                hvib[step].get(state, state).real - E0
            )

    return energies


def compute_tnacs(hvib):
    """
    Compute time-averaged nonadiabatic couplings.

    Parameters
    ----------
    hvib : list of CMATRIX

    Returns
    -------
    tnacs : ndarray
        Average NAC matrix in meV.
    """

    nstates = hvib[0].num_of_rows

    # Average over trajectory
    nacs = data_stat.cmat_stat2(hvib, 1)

    tnacs = np.zeros((nstates, nstates))

    for i in range(nstates):
        for j in range(nstates):
            tnacs[i, j] = (
                nacs.get(i, j).imag * 1000.0 / units.ev2Ha
            )

    return tnacs

In [18]:
#####################
# 3. Divide up the data (obtained from 1) into sub-trajectories. These are to be consdiered our independent nuclear sub-trajectories.
#    Here though, we just consider 1 subtrajectory, which is the entire trajectory. One could however consider more
#    The first zero below corresponds to the actual MD trajectory index (from zero). We have only 1 MD trajectory computed,
#    so this number is zero. The second number corresponds to the start time of the MD trajectory. Shown (but commented) 
#    is how one would consdier a "sub" trajectory that is the Nth MD trajectory starting from time t
#    Each trajectory, starting at time "t" is considered for length "subtraj_len"  

#####################
# 3. Divide the trajectory into subtrajectories

subtraj_time_info = [
    [0, 0], [0,299], [0, 499]  # [trajectory index (unused), starting timestep]
]

nsubtrajs = len(subtraj_time_info)
subtraj_len = 300

dt = 1.0 * units.fs2au   # 1 fs in atomic units

hvib_mixed_sd_subtrajs = []

nstates_mixed_sd = hvib_mixed_sd[0].num_of_rows

for i in range(nsubtrajs):
    hvib_mixed_sd_subtrajs.append([])


def myfunc(subtraj):

    start_time = subtraj_time_info[subtraj][1]

    print("\nSubtrajectory:", subtraj)
    print("Start time =", start_time)
    print("End time   =", start_time + subtraj_len)

    md_time = np.arange(subtraj_len) * dt * units.au2fs

    # Extract the subtrajectory
    for step in range(start_time, start_time + subtraj_len):
        hvib_mixed_sd_subtrajs[subtraj].append(hvib_mixed_sd[step])

    # Compute properties
    mixed_sd_energies = compute_state_energies_vs_time(
        hvib_mixed_sd_subtrajs[subtraj]
    )

    mixed_sd_tnacs = compute_tnacs(
        hvib_mixed_sd_subtrajs[subtraj]
    )

    mixed_sd_tau, mixed_sd_rates = decoherence_times.decoherence_times(
        hvib_mixed_sd_subtrajs[subtraj]
    )

        
    print("\nAverage state energies (relative to S0)")

    s0_energy = np.mean(mixed_sd_energies[0])

    for state in range(nstates_mixed_sd):

        avgE = np.mean(mixed_sd_energies[state] - s0_energy) * units.au2ev

        print(f"State {state:2d}: {avgE:10.5f} eV")

    # ===================== Energies =====================

    plt.figure(figsize=(3.2,2.4), dpi=300)

    plt.title("State Energies")
    plt.xlabel("Time (fs)")
    plt.ylabel("Energy (eV)")

    for state in range(nstates_mixed_sd):
        plt.plot(
            md_time,
            mixed_sd_energies[state]*units.au2ev,
            linewidth=0.8
        )

    plt.tight_layout()
    plt.savefig(f"Zn_CsPbI3_Energies_{subtraj}.png")

    # ===================== TNAC =====================

    plt.figure(figsize=(3.2,2.8), dpi=300)

    plt.title("NAC")

    plt.xlabel("State index")
    plt.ylabel("State index")

    plt.imshow(
        mixed_sd_tnacs,
        cmap="plasma",
        interpolation="nearest"
    )

    cb = plt.colorbar()
    cb.set_label("meV")

    plt.tight_layout()

    plt.savefig(f"NAC{subtraj}.png")


pool = mp.Pool(nsubtrajs)
pool.map(myfunc, range(nsubtrajs))
pool.close()
pool.join()


Subtrajectory: 0
Subtrajectory:

Subtrajectory: Start time = 10 
2
End time   =
Start time = Start time =  300299
499

End time   =End time   =  799599


Average state energies (relative to S0)
State  0:    0.00000 eV

Average state energies (relative to S0)
Average state energies (relative to S0)State  1:    1.80070 eV

State  2:    1.89023 eV
State  0:    0.00000 eV
State  0:    0.00000 eVState  3:    1.93325 eV
State  1:    1.86613 eV

State  4:    1.97679 eV
State  1:    1.79510 eV
State  2:    1.94117 eVState  5:    2.01393 eV

State  2:    1.88247 eV
State  3:    2.00224 eVState  6:    2.04307 eV

State  3:    1.94676 eV
State  4:    2.04801 eV

State  7:    2.07019 eVState  5:    2.07742 eVState  4:    1.99639 eV


State  6:    2.10547 eVState  8:    2.09605 eVState  5:    2.03622 eV


State  7:    2.12804 eVState  6:    2.06790 eVState  9:    2.11822 eV


State  8:    2.15162 eVState  7:    2.09301 eVState 10:    2.13938 eV


State  9:    2.17264 eVState  8:    2.11900 eV

Sta